# Fine-tune Wav2Vec2 (CTC) on a South African language — local RTX 3060

Tuned for a single local GPU (RTX 3060, 12GB). If you have the 6GB laptop
variant, halve `per_device_train_batch_size` and raise `gradient_accumulation_steps`
to match.

**Expected data layout**: two CSVs, `train.csv` and `eval.csv`, each with columns:
- `audio_path` — path to a 16kHz mono `.wav` file
- `sentence` — the transcript

Point these at the output of your extraction/normalization pipeline.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Config — edit these

In [ ]:
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda
BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
OUTPUT_DIR = f"./wav2vec2-{LANGUAGE}"
TRAIN_CSV = "train.csv"
EVAL_CSV = "eval.csv"

# RTX 3060 12GB: batch 4-8 with grad accumulation is a safe start.
# If you hit CUDA OOM, drop per_device_train_batch_size to 2-4.
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 4
PER_DEVICE_EVAL_BATCH = 4

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [ ]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

from datasets import load_dataset, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

## 5. Load and normalize data

In [ ]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE
)

dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [ ]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [ ]:
# for split in ["train", "dev_test"]:
#     dataset_dict[split] = dataset_dict[split].filter(
#         lambda x: x["transcript"] is not None and x["transcript"].strip() != "",
#         num_proc=4
#     )


In [ ]:
# diacritics = set()
# for split in ["train", "dev_test"]:
#     for t in dataset_dict[split]["transcript"]:
#         if t:
#             diacritics.update(c for c in t.lower() if not re.match(r"[a-z'\s0-9,\.\-\[\]]", c))

# print(sorted(diacritics))

In [ ]:
for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

## 6. Build vocabulary from your transcripts

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev_test"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

## 7. Build processor

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 8. Preprocess audio + labels

This reads every wav file — the slowest step on a local machine. `num_proc>1` can speed it up if your CPU has spare cores.

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    with processor.as_target_processor():
        batch["labels"] = processor(batch["transcript"]).input_ids
    return batch

In [ ]:

for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=1,  # bump to e.g. 4 if you have CPU cores to spare
    )

## 9. Data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 10. Metrics (WER / CER)

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

## 11. Load model

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

## 12. Training arguments

`fp16=True` roughly halves VRAM use on the 3060 — keep it on.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    logging_steps=50,
    learning_rate=3e-4,
    warmup_steps=500,
    num_train_epochs=30,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,  # trade speed for VRAM headroom
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
)

## 13. Train

On a 3060 expect roughly a few hours for a modest low-resource dataset (a few hours of audio). Watch `nvidia-smi` in a terminal alongside this if you want to confirm you're not close to OOM.

In [ ]:
trainer.train()

## 14. Save

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

## 15. Quick sanity-check inference

In [ ]:
import soundfile as sf

test_path = dataset["validation"][0] if False else None  # replace with a real wav path
# example:
# speech, sr = sf.read("some_eval_clip.wav")
# inputs = processor(speech, sampling_rate=16000, return_tensors="pt").input_values.to(model.device)
# with torch.no_grad():
#     logits = model(inputs).logits
# pred_ids = torch.argmax(logits, dim=-1)
# print(processor.batch_decode(pred_ids))